## __Aprendizaje no supervisado__

__Profesor__: Anthony D. Cho

__Ayudante__: Luis Oliveros

__Asunto__: Non-Negative Matrix Decomposition (NMF)

* __Aplicación__: Recomendación de libros

***

## Libreria

In [ ]:
import os
from tqdm import tqdm
from time import time
import matplotlib.pyplot as plt
from pandas import read_csv, DataFrame, set_option
set_option("display.max_columns", None)

from sklearn.decomposition import NMF

<center>
  <img src=https://thumbs.dreamstime.com/b/flying-magic-books-library-367534733.jpg width=800>
</center>

__Dataset:__

La dataset contine información relacionado a ratings de libros y está disponible en el repositorio [Kaggle: Book Recommendation Dataset](https://www.kaggle.com/datasets/arashnic/book-recommendation-dataset). A continuación, se describe la información segmentada:

* __Users__<br>
Contains the users. Note that user IDs (User-ID) have been anonymized and map to integers. Demographic data is provided (Location, Age) if available. Otherwise, these fields contain NULL-values.

* __Books__<br>
Books are identified by their respective ISBN. Invalid ISBNs have already been removed from the dataset. Moreover, some content-based information is given (Book-Title, Book-Author, Year-Of-Publication, Publisher), obtained from Amazon Web Services. Note that in case of several authors, only the first is provided. URLs linking to cover images are also given, appearing in three different flavours (Image-URL-S, Image-URL-M, Image-URL-L), i.e., small, medium, large. These URLs point to the Amazon web site.

* __Ratings__<br>
Contains the book rating information. Ratings (Book-Rating) are either explicit, expressed on a scale from 1-10 (higher values denoting higher appreciation), or implicit, expressed by 0.

<p style="color:red"> <b>Instrucciones para usuarios que usan Windows: </b></p>

* Descargar la data pulsando [¡¡AQUI!!](https://mayorcl-my.sharepoint.com/:u:/g/personal/anthony_cho_umayor_cl/Eetj7dIQlHhMjuZf6OkdHscBFThpSeuEuNt4vRwEKOW5uw?download=1)
* Descomprimir el archivo __Book-Crossing.zip__.
* Dejar los archivos csv (__BX-Book-Ratings.csv, BX-Books.csv, BX-Users.csv__) junto con el script.
* Ya con eso, debería poder ejecutar el script sin ningun problema.

In [ ]:
## Solo para Linux y MacOS
if os.name == 'posix':
    if not os.path.exists("Book-Crossing.zip"):

        ## Eliminar los archivos relacionados
        !rm BX-Book-Ratings.csv BX-Books.csv BX-Users.csv

        ## Download files
        !wget https://mayorcl-my.sharepoint.com/:u:/g/personal/anthony_cho_umayor_cl/Eetj7dIQlHhMjuZf6OkdHscBFThpSeuEuNt4vRwEKOW5uw?download=1

        ## renombrar archivo
        !mv Eetj7dIQlHhMjuZf6OkdHscBFThpSeuEuNt4vRwEKOW5uw?download=1 Book-Crossing.zip

        ## Descomprimir el .zip
        !unzip Book-Crossing.zip
else:
    print('Seguir las instrucciones para usuarios de Windows mencionado arriba.')

## Carga de datos

In [ ]:
## Carga de información de los libros
books_raw = read_csv('BX-Books.csv', sep=';', encoding='latin-1', usecols=range(5))
books_raw.head()

In [ ]:
## Carga de información de los usuarios-libros con sus respectivos ratings
ratings_raw = read_csv('BX-Book-Ratings.csv', sep=';', encoding='latin-1')
ratings_raw.head()

In [ ]:
ratings_raw['Book-Rating'].describe()

#### Preprocesamiento de los datos

* Se filtran los datos para los libros que tengan mas de 20 evaluaciones y aquellos usuarios que han hecho mas de 3 revisiones

* Se reemplazan los ISBN por los titulos de los libros.

In [ ]:
## Copia del dataframe
ratings = ratings_raw.copy()

## Filtramos los libros que tiene más de 20 ratings
book_rating_group = ratings.groupby(['ISBN']).count()
book_rating_group = book_rating_group[book_rating_group['Book-Rating']>20]
ratings = ratings[ratings['ISBN'].isin(book_rating_group.index)]

## Filtramos los usuarios que han evaluado más de tres libros.
user_rating_group = ratings.groupby(['User-ID']).count()
user_rating_group = user_rating_group[user_rating_group['Book-Rating']>3]
ratings = ratings[ratings['User-ID'].isin(user_rating_group.index)]

display(ratings.head())

In [ ]:
## Reemplazo de cada ISBN por su titulo del libro.
user_book = ratings.merge(right=books_raw[['ISBN', 'Book-Title']], on='ISBN').drop(columns=['ISBN'])

## Eliminar los duplicados
user_book = user_book.drop_duplicates(subset=['User-ID', 'Book-Title'])
user_book

Se convierte a una tabla de frecuencia con Titulos vs ID de usuarios

In [ ]:
## Transformación de la representación de información
user_book_table = user_book.pivot(index='Book-Title', columns='User-ID', values='Book-Rating')
print('(shape) user_book_table: {}'.format(user_book_table.shape))
user_book_table.head(5)

Rellenado de los NaN usando el promedio por usuario

In [ ]:
## Calculo del promedio por usuario (por columna)
mean_user = user_book_table.mean(axis=0)

## Centrar los datos en cero
user_book_table_fill = user_book_table.sub(mean_user)

## Se reemplaza los NaN por cero
user_book_table_fill = user_book_table_fill.fillna(0)

## Se descentraliza los datos con el promedio por usuario
user_book_table_fill = user_book_table_fill.sub(-mean_user)

In [ ]:
user_book_table_fill.head()

## Construcción del modelo

Existen múltiples hiperparámetros para el modelo NMF:<br>

```{python}
    nmf_model = NMF(n_components=None,
                    tol=0.0001,
                    max_iter=200,
                    random_state=None)
```

| Hiperparámetros | Descripción |
|-----------------|-------------|
| __n_components__  | número de componentes (por defecto, None (todos)).|
| __tol__ | margen de error entre iteraciones |
| __max_iter__ | número máximo de iteraciones (por defecto, 200) |
| __random_state__ | semilla de aleatoriedad para inicialización de los puntos en el espacio reducido.|

***

Para ajuste y transformación de los datos se emplean las siguientes funciones:

* __fit(X)__ : entrena el modelo usando un conjunto de datos.

* __transform(X)__: transforma un conjunto de datos al espacio reducido.

* __fit_transform(X)__: entrena el modelo y transforma un conjunto de datos al espacio reducido.

* inverse_transform(X)__: transforma un conjunto de datos del espacio reducido al espacio original.

***


| Atributos | Descripción |
|-----------|-------------|
| components_ | Retorna la matriz U que representa la muestras en el espacio reducido |
| reconstruction_err_ | retorna el error de reconstrucción basada en un métrica (Por defecto, norma de frobenius) |


In [ ]:
## Duración: 7 min aprox en colab

start = time()

## Instancia del modelo
model = NMF(n_components=4, max_iter=20000, random_state=9001)

## Ajuste del modelo
model.fit(user_book_table_fill)

## Retorno de la matriz U
U = DataFrame(model.transform(user_book_table_fill), 
              index=user_book_table_fill.index)
print('(shape) U: {}'.format(U.shape))

## Retorno de la matriz V
V = DataFrame(model.components_, 
              columns=user_book_table_fill.columns)
print('(shape) V: {}'.format(V.shape))

## Reconstrucción de los datos
R = DataFrame(U.dot(V))
R.columns = user_book_table_fill.columns.values
R.index = user_book_table_fill.index
print('(shape) R: {}'.format(R.shape))

timeUp = time()
print('Time spent[s]: {}'.format(timeUp - start))
R.head(5)


In [ ]:
## Mostrar error obtenido
model.reconstruction_err_

In [ ]:
user_book_table_fill.shape

In [ ]:
U

In [ ]:
V

## Análisis de las matrices U y V

In [ ]:
## Matriz U: Titulos vs variables latente
U.T

In [ ]:
U.describe()

In [ ]:
U.loc[U[3]>0.09][3].index.to_list()

Se podría decir que las variables latente representarían:

| **Variable latente** | **Contenido**                                                                                                                                                 | **Temática**                                                        |
|----------------------|---------------------------------------------------------------------------------------------------------------------------------------------------------------|---------------------------------------------------------------------|
| 0                    | Drama Familiar y Social, Suspenso y Thriller, Romance, Fantástico y Sobrenatural                                                                              | Ficción Contemporánea, Misterio y Supervivencia Personal (Th: 1.93) |
| 1                    | Literatura Clásica y Contemporánea, Misterio y Thriller, Realismo y Drama, Fantasía y Ciencia Ficción, Literatura Juvenil y Fantasía.                         | Narrativas Intemporales: Clásicos, Misterio y Realismo (Th: 0.55)   |
| 2                    | Suspenso y Thriller, Drama y Realismo Contemporáneo, Fantasía y Ciencia Ficción Contemporánea, Misterio y Crimen, Literatura Juvenil y de Crecimiento Persona | Fantasía Contemporánea (Th: 0.1)                                    |
| 3                    | Ficción Contemporánea y Drama, Misterio y Thriller, Fantástica y Ciencia Ficción, Autoayuda, Literatura Juvenil y Fantasía.                                   | Dramas misteriosas (Th: 0.09)                                       |


In [ ]:
## Matriz V: variables latente vs users
V

In [ ]:
V.T.describe()

#### Aplicación: recomendación

In [ ]:
## ID del usuario
user_id = 268711

## Libros que ya ha revisado por el usuario
books_user = user_book_table[user_id].dropna().index
books_user

In [ ]:
## Buscamos los indices que coinciden con la lista de los libros revisado por el usuario
index = R[user_id].index.isin(books_user)

## Extraemos los libros que no ha revisado por el usuario
noRevs = R[user_id][~index].sort_values(ascending=False)
print('(No revs) {}'.format(noRevs.shape))

In [ ]:
## Mostrar 10 recomendaciones
n = 10

for i in range(n):
    print(noRevs.index[i])

## Una estratégia para estimar el número de componentes

In [ ]:
## Duración: 44 min aprox en colab

## Almacenador de errores
error = []

## Lista de posibles número de componenetes
n_list = list(range(2, 61))

for i in tqdm(n_list):

    ## Instancia del modelo
    model = NMF(n_components=i, max_iter=20000, random_state=9001)

    ## Ajuste del modelo
    model.fit(user_book_table_fill)

    ## Almacenar el error de reconstrucción
    error.append(model.reconstruction_err_)

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(n_list, error)
plt.ylabel('Error')
plt.xlabel('Número de componentes')
plt.tight_layout()
plt.show()